# 04 — SVM (Linear) From Scratch vs Sklearn
Implementasi Linear Soft-Margin SVM from scratch, perbandingan dengan sklearn.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.utils import set_seed, save_model
from src.data import load_train
from src.cleaning import DataCleaner
from src.preprocessing import Preprocessor
from src.algorithms.svm import LinearSVMScratch
from src.optimizers import GradientDescent, Adam
from src.evaluation import cross_validate, macro_f1_score, stratified_k_fold_indices
from src.sklearn_baselines import get_sklearn_svm
from src.visualization import plot_loss_curves
from src import config

set_seed(42)

In [ ]:
# Load & preprocess
train_df = load_train()
cleaner = DataCleaner()
train_clean = cleaner.fit_transform(train_df)
preprocessor = Preprocessor()
X_train, y_train = preprocessor.fit_transform(train_clean)
print(f'X_train: {X_train.shape}')

In [ ]:
# Compare GD vs Adam convergence for SVM
folds = stratified_k_fold_indices(y_train, n_folds=5)
tr_idx, val_idx = folds[0]
X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
X_val, y_val = X_train[val_idx], y_train[val_idx]

# SVM with GD
svm_gd = LinearSVMScratch(
    C=1.0, max_iter=500, batch_size=512,
    class_weight='balanced', optimizer=GradientDescent(lr=0.001)
)
svm_gd.fit(X_tr, y_tr)

# SVM with Adam
svm_adam = LinearSVMScratch(
    C=1.0, max_iter=500, batch_size=512,
    class_weight='balanced', optimizer=Adam(lr=0.001)
)
svm_adam.fit(X_tr, y_tr)

print(f'GD - Final loss: {svm_gd.loss_history[-1]:.4f}, Macro F1: {macro_f1_score(y_val, svm_gd.predict(X_val)):.4f}')
print(f'Adam - Final loss: {svm_adam.loss_history[-1]:.4f}, Macro F1: {macro_f1_score(y_val, svm_adam.predict(X_val)):.4f}')

In [ ]:
# Plot SVM loss convergence
import os
plot_loss_curves(
    {'Gradient Descent': svm_gd.loss_history, 'Adam': svm_adam.loss_history},
    title='SVM: Loss Convergence — GD vs Adam',
    save_path=os.path.join(config.FIGURES_DIR, 'svm_convergence_gd_vs_adam.png')
)

In [ ]:
# Cross-validate SVM from scratch (with threshold tuning)
def svm_factory():
    return LinearSVMScratch(
        C=1.0, max_iter=500, batch_size=512,
        class_weight='balanced', optimizer=Adam(lr=0.001)
    )

svm_cv = cross_validate(
    svm_factory, X_train, y_train, n_folds=5,
    tune_threshold=True, use_decision_function=True
)
print(f'SVM From-Scratch (Adam + threshold tuning):')
print(f'  Mean Macro F1: {svm_cv["mean_f1"]:.4f} +/- {svm_cv["std_f1"]:.4f}')

In [ ]:
# Sklearn baseline
svm_sk_cv = cross_validate(get_sklearn_svm, X_train, y_train, n_folds=5)
print(f'Sklearn LinearSVC:')
print(f'  Mean Macro F1: {svm_sk_cv["mean_f1"]:.4f} +/- {svm_sk_cv["std_f1"]:.4f}')

In [ ]:
# Save model
svm_final = svm_factory()
svm_final.fit(X_train, y_train)
save_model(svm_final, 'linear_svm_adam.pkl')
print('Model saved.')